# Deep Learning Project 1: a ResNet under 5M parameters for CIFAR-10

This notebook reproduces our results end to end: the data, the final architecture and its parameter count, training curves, validation accuracy, an ablation comparison, and the Kaggle submission. Training itself runs through the same package code as the command line (`cifar_resnet.train`), so the numbers here match `runs/`.

## 1. Setup

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))  # works with or without `pip install -e .`

from cifar_resnet.checkpoint import load_model
from cifar_resnet.config import load_config
from cifar_resnet.data import CLASSES, build_val_split, load_split
from cifar_resnet.engine import predict_logits
from cifar_resnet.models import build_model, count_parameters
from cifar_resnet.params import layer_table
from cifar_resnet.utils import get_device

CONFIG = 'configs/student.yaml'  # final architecture and recipe
FINAL_RUN = 'student_full'        # same recipe retrained on all 50k images
RUN_TRAINING = False            # True trains here; False loads results from runs/

cfg = load_config(CONFIG)
device = get_device()
print(f'torch {torch.__version__} | device {device}')

## 2. Data

In [ ]:
train_split = load_split(cfg.data.root, 'train', cfg.data.cache_dir)
test_split = load_split(cfg.data.root, 'test', cfg.data.cache_dir)
print('train', train_split.images.shape, '| test', test_split.images.shape)
print(pd.Series(train_split.labels).map(dict(enumerate(CLASSES))).value_counts().sort_index())

fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for c, col in enumerate(axes.T):
    idx = np.flatnonzero(train_split.labels == c)[:2]
    for ax, i in zip(col, idx):
        ax.imshow(train_split.images[i]); ax.axis('off')
    col[0].set_title(CLASSES[c], fontsize=9)
plt.tight_layout(); plt.show()

## 3. Model and parameter count

In [ ]:
model = build_model(cfg.model)
print(model)
print(layer_table(model))
try:
    from torchsummary import summary
    summary(model, (3, 32, 32), device='cpu')
except ImportError:
    pass
N_PARAMS = count_parameters(model)
assert N_PARAMS <= 5_000_000
print(f'Trainable parameters: {N_PARAMS:,}  (budget 5,000,000)')

## 4. Training

Set `RUN_TRAINING = True` to train here (a GPU is strongly recommended). Otherwise this loads the run produced by `cifar-train --config configs/student.yaml`.

In [ ]:
if RUN_TRAINING:
    from cifar_resnet.train import train
    summary_val = train(cfg)
run_dir = cfg.run_dir
summary_val = json.loads((run_dir / 'summary.json').read_text())
summary_val

## 5. Training curves

In [ ]:
m = pd.read_csv(run_dir / 'metrics.csv')
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(m.epoch, m.train_loss, label='train'); ax[0].plot(m.epoch, m.val_loss, label='val')
ax[0].set(title='Loss', xlabel='epoch'); ax[0].legend()
ax[1].plot(m.epoch, m.train_acc, label='train (augmented)')
ax[1].plot(m.epoch, m.val_acc, label='val')
if 'ema_val_acc' in m and m.ema_val_acc.notna().any():
    ax[1].plot(m.epoch, m.ema_val_acc, label='val (EMA weights)')
ax[1].set(title='Accuracy', xlabel='epoch', ylim=(0.5, 1)); ax[1].legend()
ax[2].plot(m.epoch, m.lr); ax[2].set(title='Learning rate', xlabel='epoch')
plt.tight_layout(); plt.show()

## 6. Validation accuracy of the best checkpoint

In [ ]:
best, best_cfg, ckpt = load_model(run_dir / 'best.pt', device)
val_ds = build_val_split(best_cfg)
loader = torch.utils.data.DataLoader(val_ds, batch_size=512)
logits, labels = predict_logits(best, loader, device, tta=False)
logits_tta, _ = predict_logits(best, loader, device, tta=True)
acc = (logits.argmax(1) == labels).float().mean().item()
acc_tta = (logits_tta.argmax(1) == labels).float().mean().item()
print(f'Validation accuracy: {acc:.4f} | with flip TTA: {acc_tta:.4f} '
      f'({len(val_ds)} held-out images, epoch {ckpt["epoch"]}, {ckpt["source"]} weights)')

pred = logits_tta.argmax(1).numpy(); y = labels.numpy()
cm = np.zeros((10, 10), dtype=int)
np.add.at(cm, (y, pred), 1)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].bar(CLASSES, cm.diagonal() / cm.sum(1)); ax[0].set(title='Per-class accuracy', ylim=(0.7, 1))
ax[0].tick_params(axis='x', rotation=45)
ax[1].imshow(cm, cmap='Blues'); ax[1].set(title='Confusion matrix', xlabel='predicted', ylabel='true')
ax[1].set_xticks(range(10), CLASSES, rotation=90); ax[1].set_yticks(range(10), CLASSES)
for i in range(10):
    for j in range(10):
        ax[1].text(j, i, cm[i, j], ha='center', va='center', fontsize=7)
plt.tight_layout(); plt.show()

## 7. Comparing runs (architectures and ablations)

In [ ]:
rows = [json.loads(p.read_text()) for p in sorted(Path('runs').glob('*/summary.json'))]
runs = pd.DataFrame(rows)
cols = ['run_name', 'params', 'epochs', 'train_images', 'best_val_acc', 'best_val_acc_tta', 'train_time_min']
runs[[c for c in cols if c in runs]].sort_values('best_val_acc', ascending=False)

## 8. Final model and Kaggle submission

The final model is the chosen config retrained on all 50,000 training images (`scripts/hpc/pipeline.sh` or `bash scripts/run_final.sh configs/student.yaml`).

In [ ]:
from cifar_resnet.predict import predict

final_ckpt = Path('runs') / FINAL_RUN / 'best.pt'
sub = predict(final_ckpt, f'submissions/{FINAL_RUN}.csv', tta=True)
final_model, _, _ = load_model(final_ckpt, 'cpu')
print(sub.head())
print(sub.label.map(dict(enumerate(CLASSES))).value_counts().sort_index())

In [ ]:
KAGGLE_PUBLIC_ACC = None   # fill in from the leaderboard after submitting
KAGGLE_PRIVATE_ACC = None  # fill in after the competition closes

print('=' * 60)
print(f'Final model parameters : {count_parameters(final_model):,}')
print(f'Validation accuracy    : {acc_tta:.4f} (held-out 5k, {cfg.run_name})')
print(f'Kaggle test accuracy   : public {KAGGLE_PUBLIC_ACC}, private {KAGGLE_PRIVATE_ACC}')
print('=' * 60)